In [39]:
import boto3
import os
from dotenv import load_dotenv

load_dotenv()
access_key_id = os.getenv("ACCESS_KEY_ID")
secret_access_key = os.getenv("SECRET_ACCESS_KEY")
minio_url = "http://" + os.getenv("S3_API_ENDPOINT")


minio_client = boto3.client(
    "s3",
    aws_access_key_id=access_key_id,
    aws_secret_access_key=secret_access_key,
    endpoint_url=minio_url
)

new_bucket = "exploitation-zone-task2-images-text"
try:
    minio_client.create_bucket(Bucket=new_bucket)
except (minio_client.exceptions.BucketAlreadyExists, minio_client.exceptions.BucketAlreadyOwnedByYou):
    print(f"Bucket '{new_bucket}' already exists")

Bucket 'exploitation-zone-task2-images-text' already exists


In [ ]:
from chromadb.utils.embedding_functions import EmbeddingFunction
from transformers import ClapProcessor, ClapModel
import torch
import soundfile as sf
import os
import io
import chromadb

# --- CONFIGURACIÓ DE MINIO I CHROMADB ---
client = chromadb.HttpClient(host="localhost", port=8000)
paginator = minio_client.get_paginator('list_objects_v2')

trusted_zone = "trusted-zone"
exploitation_zone = "exploitation-zone-task2-images-text"
name_collection1 = "multimodal_collection_images"
name_collection2 = "multimodal_collection_text"
name_collection3 = "multimodal_collection_audios"

# ==========================
#     MODEL CLAP (AUDIO)
# ==========================
class CLAPEmbeddingFunction(EmbeddingFunction):
    """Embedding function per a àudio usant el model oficial de HuggingFace."""
    def __init__(self):
        print("⬇️ Carregant model 'laion/clap-htsat-fused' des de HuggingFace...")
        self.processor = ClapProcessor.from_pretrained("laion/clap-htsat-fused")
        self.model = ClapModel.from_pretrained("laion/clap-htsat-fused")
        self.model.eval()
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(self.device)
        print("✅ Model CLAP carregat correctament!")

    def __call__(self, items):
        embeddings = []
        for path in items:
            try:
                audio, sr = sf.read(path)
                inputs = self.processor(audio=audio, sampling_rate=sr, return_tensors="pt")
                inputs = {k: v.to(self.device) for k, v in inputs.items()}
                with torch.no_grad():
                    emb = self.model.get_audio_features(**inputs)
                embeddings.append(emb.squeeze().cpu().numpy())
            except Exception as e:
                print(f"⚠️ Error processant {path}: {e}")
        return embeddings


# ==========================
#     CREACIÓ DE COL·LECCIÓ
# ==========================
try:
    client.delete_collection(name=name_collection3)
except Exception:
    pass

try:
    print("Creant la col·lecció d'àudio amb CLAP (HuggingFace)...")
    audio_embedding_function = CLAPEmbeddingFunction()
    audio_col = client.get_or_create_collection(
        name=name_collection3,
        embedding_function=audio_embedding_function
    )
except Exception as e:
    print(f"❌ Error creant la col·lecció d'àudio: {e}")
    exit(1)

# ==========================
#     PROCESSAMENT D'ÀUDIOS
# ==========================
for page in paginator.paginate(Bucket=trusted_zone, Prefix="audio/"):
    for obj in page.get("Contents", []):
        key = obj.get("Key", "")
        if not key.lower().endswith((".wav", ".mp3", ".flac", ".ogg")):
            continue

        try:
            # Descarrega de MinIO
            response = minio_client.get_object(Bucket=trusted_zone, Key=key)
            audio_data = response['Body'].read()

            # Desa temporalment per processar
            temp_path = f"/tmp/{os.path.basename(key)}"
            with open(temp_path, "wb") as f:
                f.write(audio_data)

            # ✅ Genera embedding manualment
            embedding = audio_embedding_function([temp_path])[0]

            # ✅ Desa directament embedding (NO uris)
            audio_col.add(
                ids=[key],
                embeddings=[embedding],
                metadatas=[{"bucket": trusted_zone, "key": key, "type": "audio"}]
            )

            # Copia a zona d'explotació
            minio_client.copy_object(
                Bucket=exploitation_zone,
                CopySource={'Bucket': trusted_zone, 'Key': key},
                Key=key
            )

            print(f"✅ Processat i afegit: {key}")

        except Exception as e:
            print(f"⚠️ Error processant {key}: {e}")

# ==========================
#     RESUM FINAL
# ==========================
print("\n✅ Col·lecció d'àudio creada correctament!")
print("Nombre total d'àudios:", len(audio_col.get()['ids']))


Creant la col·lecció d'àudio amb CLAP (HuggingFace)...
⬇️ Carregant model 'laion/clap-htsat-fused' des de HuggingFace...
✅ Model CLAP carregat correctament!
✅ Processat i afegit: audio/answer_0.mp3
✅ Processat i afegit: audio/answer_1.mp3
✅ Processat i afegit: audio/answer_10.mp3
✅ Processat i afegit: audio/answer_11.mp3
✅ Processat i afegit: audio/answer_12.mp3
✅ Processat i afegit: audio/answer_13.mp3
✅ Processat i afegit: audio/answer_14.mp3
✅ Processat i afegit: audio/answer_15.mp3
✅ Processat i afegit: audio/answer_16.mp3
✅ Processat i afegit: audio/answer_17.mp3
✅ Processat i afegit: audio/answer_18.mp3
✅ Processat i afegit: audio/answer_19.mp3
✅ Processat i afegit: audio/answer_2.mp3
✅ Processat i afegit: audio/answer_20.mp3
✅ Processat i afegit: audio/answer_21.mp3
✅ Processat i afegit: audio/answer_22.mp3
✅ Processat i afegit: audio/answer_23.mp3
✅ Processat i afegit: audio/answer_24.mp3
✅ Processat i afegit: audio/answer_25.mp3
✅ Processat i afegit: audio/answer_26.mp3
✅ Proc

In [43]:
import io
import soundfile as sf
import numpy as np
import IPython.display as ipd

def cerca_audio_per_text_i_reprodueix(consulta_text: str, top_k: int = 1):
    """
    Cerca l'àudio més semblant al text i el reprodueix.
    """
    print(f"\n🔎 Buscant àudios més semblants a: \"{consulta_text}\" ...")

    # --- Embedding del text ---
    inputs = audio_embedding_function.processor(
        text=[consulta_text],
        return_tensors="pt"
    )
    inputs = {k: v.to(audio_embedding_function.device) for k, v in inputs.items()}
    with torch.no_grad():
        query_emb = audio_embedding_function.model.get_text_features(**inputs)
    query_emb = query_emb.squeeze().cpu().numpy()

    # --- Consulta a la col·lecció ---
    results = audio_col.query(
        query_embeddings=[query_emb],
        n_results=top_k
    )

    if len(results["ids"][0]) == 0:
        print("⚠️ No s'ha trobat cap resultat.")
        return None

    print("\n🎧 Resultats:")
    for i, (id_, meta) in enumerate(zip(results["ids"][0], results["metadatas"][0])):
        print(f"{i+1}. ID: {id_}")
        print(f"   → Metadata: {meta}")
        print(f"   → Distància (similitud inversa): {results['distances'][0][i]:.4f}")

    # --- Reprodueix el primer resultat ---
    best_key = results["ids"][0][0]
    print(f"\n▶️ Reproduint el millor resultat: {best_key}")

    # Descarrega des de MinIO
    response = minio_client.get_object(Bucket=trusted_zone, Key=best_key)
    audio_bytes = response['Body'].read()

    # Carrega amb soundfile
    audio_array, sr = sf.read(io.BytesIO(audio_bytes))

    # Mostra reproductor dins Jupyter (si n'hi ha)
    try:
        display(ipd.Audio(audio_array, rate=sr))
    except:
        print("🎵 (No es pot mostrar el reproductor interactiu en aquest entorn.)")

    return results

# Exemple d'ús
cerca_audio_per_text_i_reprodueix("un bollito")



🔎 Buscant àudios més semblants a: "un bollito" ...

🎧 Resultats:
1. ID: audio/answer_22.mp3
   → Metadata: {'key': 'audio/answer_22.mp3', 'bucket': 'trusted-zone', 'type': 'audio'}
   → Distància (similitud inversa): 1.5053

▶️ Reproduint el millor resultat: audio/answer_22.mp3


{'ids': [['audio/answer_22.mp3']],
 'distances': [[1.5052805]],
 'embeddings': None,
 'metadatas': [[{'key': 'audio/answer_22.mp3',
    'bucket': 'trusted-zone',
    'type': 'audio'}]],
 'documents': [[None]],
 'uris': None,
 'data': None,
 'included': ['metadatas', 'documents', 'distances']}